# Fine-tune DistilBERT for product-review sentiment

Run on Colab with **Runtime → Change runtime type → T4 GPU**. The default config (60k reviews, 2 epochs, fp16) takes roughly 2–2.5 h on a T4; lower `--train_samples` for a faster first pass.

Pipeline: amazon_polarity → disjoint train / val / test → fine-tune → evaluate on the test set **once** → acceptance gates (accuracy ≥ 0.92, F1 macro ≥ 0.91, ROC-AUC ≥ 0.95) → save model + drift reference.

In [ ]:
REPO_URL = 'https://github.com/Srujan641/Sentiment-Analysis-API.git'
!git clone $REPO_URL sentiment-api
%cd sentiment-api
!pip install -q -r requirements-train.txt

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: enable a T4 runtime')

## Optional: Weights & Biases
Log in to track the run (and sweeps) on your W&B dashboard. Skip this cell to train without tracking.

In [ ]:
import wandb
wandb.login()

## Option A: single training run

In [ ]:
!python -m scripts.train --fp16 --wandb --log_artifact \
    --train_samples 60000 --val_samples 5000 --test_samples 10000 \
    --learning_rate 3e-5 --batch_size 32 --epochs 2 --warmup_ratio 0.1

## Option B: Bayesian sweep (10 runs), then retrain the winner
Pick the best run's hyperparameters from the sweep page, then run Option A with them. Sweep runs select on **validation** F1 macro; the test set is only touched by the final run.

In [ ]:
!python -m scripts.sweep --count 10

## Inspect the result

In [ ]:
import json
print(json.dumps(json.load(open('models/distilbert-sentiment/metrics.json')), indent=2))

In [ ]:
from transformers import pipeline
clf = pipeline('text-classification', model='models/distilbert-sentiment', top_k=None)
for t in ['Best product I have ever bought!', 'This is not what I would call a good product.',
          'Arrived on time and looks nice, but the battery dies in an hour.']:
    print(t, '->', clf(t)[0])

## Ship it
Download the folder (or pull the W&B artifact on your server with `python -m scripts.download_model --wandb <entity>/<project>/distilbert-sentiment:latest`) and place it in `models/distilbert-sentiment/`.

In [ ]:
!zip -qr distilbert-sentiment.zip models/distilbert-sentiment
from google.colab import files
files.download('distilbert-sentiment.zip')